## Programming for Data Science 
### Project Notebook: "Where should I live?" 
#### Group Members:
- Afonso Fernandes / 20241710
- Lourenço Lima / 20241711
- Pedro Jorge / 20241819
- David Morais / 20241759
## Project Repository
GitHub Repository:  
https://github.com/afonsolince06/-Where-should-I-live-PDS-Project


### Introduction

In this part of the project, the goal is to build an interactive map of Europe that allows users to explore key information about major European cities. The task combines web scraping, data integration, and geospatial visualization to create an informative and interactive tool.

To accomplish this, we will:

-Scrape the geographical coordinates of each city directly from the Wikipedia Main Page, ensuring accuracy and consistency with the provided dataset.

-Match the scraped coordinates with the dataset entries so that each city is correctly assigned to its corresponding country, population, average salary, and cost of living.

-Use the cleaned and enriched dataset to construct an interactive map of Europe, where each city can be clicked or hovered over to display its relevant information.

By the end of this section, we will have a fully functional map that visually represents European cities and provides meaningful insights through an intuitive interface. This builds on the skills developed earlier in the project and introduces new concepts in geospatial data handling and visualization.

### Importing the necessary libraries for webscraping and map building.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import selenium
import requests
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

### Importing the already cleaned dataset

In [2]:
city_data = pd.read_csv('city_data_cleaned.csv')
city_data_coords = city_data.copy()
#city_data_coords['Latitude'] = np.nan
#city_data_coords['Longitude'] = np.nan

### Web Scrapping and Interative Map
In this section, Selenium is used to automatically retrieve the geographic coordinates (latitude and longitude) of European cities from the cleaned dataset `city_data_coords` using information available on Wikipedia. The process begins with the implementation of the function `get_city_coordinates`, which opens a web browser and navigates to the Wikipedia main page. For each city in the dataset, a search is performed—optionally including the country name to improve accuracy—followed by navigation to the corresponding city article, where the geographic coordinates are extracted.

The retrieved coordinates are subsequently stored in the dataset through the function `fill_dataset_coords`. Since the coordinates obtained from Wikipedia are provided in DMS (Degrees, Minutes, Seconds) format, an additional function, `dms_to_decimal`, is applied to convert them into decimal degrees. These processed coordinates are then used as input for the construction of an interactive map visualization, allowing for a clear geographical representation of the analyzed cities.

In [5]:
wikipedia_url = "https://en.wikipedia.org/wiki/Main_Page"


def get_city_coordinates(city_name):
    """
    Starts on the Wikipedia main page and searches for the given city name.

    Parameters:
    city_name (str): The name of the city to search for.

    Returns:
    The city page URL.
    """

    if city_name == 'Rotterdam': # special case for Rotterdam
        try:
            # navigate to Wikipedia main page
            driver.get(wikipedia_url)

            # find the search input box and enter the city name
            search_input = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.ID, "searchInput")))
            search_input.send_keys(city_name)
            search_input.send_keys(Keys.RETURN)

        
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, "firstHeading"))) # wait for page to load
            lat = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//span[@class='latitude']")))
            latitude = driver.execute_script("return arguments[0].textContent;", lat)
            long = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//span[@class='longitude']")))
            longitude = driver.execute_script("return arguments[0].textContent;", long)
            return (latitude.strip(), longitude.strip())
        
        finally:
            driver.quit() # close the browser

    else: # general case for other cities
        # set up the Selenium WebDriver 
        options = webdriver.ChromeOptions()
        driver = webdriver.Chrome(options=options)
        driver.maximize_window()
        
        try:
            # navigate to Wikipedia main page
            driver.get(wikipedia_url)
            # find the search input box and enter the city name
            search_input = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.ID, "searchInput")))
            search_input.send_keys(city_name)
            search_input.send_keys(Keys.RETURN)
          
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, "firstHeading"))) # wait for page to load
            city_latitude = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "latitude")))
            city_longitude = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "longitude")))
            return (city_latitude.text, city_longitude.text)
            
        finally:
            driver.quit() # close the browser
    


def fill_dataset_coords(df):
    """
    Fills the dataset with latitude and longitude coordinates for each city.
    Parameters:
    df (DataFrame): The DataFrame containing city data.
    Returns:
    The updated DataFrame with latitude and longitude columns filled.
    """

    for city in df['City']:
        try: # attempt to get coordinates using "City, Country" format first
            df.loc[df['City'] == city, ['Latitude', 'Longitude']] = get_city_coordinates(f"{city}, {df.loc[city_data_coords['City'] == city, 'Country'].values[0]}")
        except Exception: # if that fails, try just the city name
            df.loc[df['City'] == city, ['Latitude', 'Longitude']] = get_city_coordinates(city)
    return df





# # filling the dataset with coordinates
# fill_dataset_coords(city_data_coords)

# # as we were not able to scrapte the coordinates for Rotterdam, we will fill it manually just so we have it in the map
# display(city_data_coords.tail(40))
# city_data_coords.loc[city_data_coords['City'] == 'Rotterdam', ['Latitude', 'Longitude']] = ('51°55′N', '4°29′E')
# display(city_data_coords.loc[city_data_coords['City'] == 'Rotterdam', ['City','Latitude', 'Longitude']])

# city_data_coords.to_csv('city_data_with_coords.csv', index=False)

# safety net in case we need to reload the dataset with coordinates after deleting them by accident
city_data_coords = pd.read_csv('city_data_with_coords.csv')


# # Iterative Map Building and visualization



def dms_to_decimal(dms_coords):
    """
    Converts the coordinates from the DMS (Degrees, Minutes, Seconds) format that they were scraped in to a decimal format to be used in the map visualization.
    
    Parameters:
    dms_coords (str): The coordinates in DMS format.
    
    Returns:
    The coordinates in decimal format.
    """
    dms_parts = re.split('[°′″NEW]', dms_coords)
    degrees = float(dms_parts[0]) # degrees part is always present and it's the first element
    minutes = float(dms_parts[1]) if len(dms_parts) > 1 and dms_parts[1] else 0 # minutes part may be missing, it's the second element
    seconds = float(dms_parts[2]) if len(dms_parts) > 2 and dms_parts[2] else 0 # seconds part may be missing, it's the third element

    decimal_coords = degrees + (minutes / 60) + (seconds / 3600) # formula to convert DMS to decimal

    if 'S' in dms_coords or 'W' in dms_coords:
        decimal_coords = -decimal_coords # southern and western hemispheres are negative

    return decimal_coords

city_data_coords['Latitude'] = city_data_coords['Latitude'].apply(dms_to_decimal)
city_data_coords['Longitude'] = city_data_coords['Longitude'].apply(dms_to_decimal)

display(city_data_coords[['City','Latitude', 'Longitude']].tail(51))

map_df = city_data_coords.dropna(subset=['Latitude', 'Longitude'])

fig_map = px.scatter_map(
        map_df,
        lat="Latitude",
        lon="Longitude",
        hover_name="City",
        hover_data={
            "Latitude": False, "Longitude": False,
            "Country": True,
            "Population": ':,.0f', 
            "Average Monthly Salary": ':,.0f €', 
            "Average Cost of Living": ':,.0f €'  
        },
        color="Average Monthly Salary", 
        size="Population", 
        size_max=25,             
        color_continuous_scale="Turbo",
        zoom=3, 
        height=700,
        title="Iterative Map of Cities with Population, Average Monthly Salary, and Cost of Living"
    )

fig_map.update_layout(mapbox_style="open-street-map")
fig_map.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_map.show()

,City,Latitude,Longitude
33,Thessaloniki,40.640278,22.935556
34,Budapest,47.492500,19.051389
35,Debrecen,47.531667,21.624444
36,Miskolc,48.104167,20.791667
37,Cork,51.897222,-8.470000
38,Dublin,53.350000,-6.260278
39,Florence,43.771389,11.254167
40,Milan,45.466944,9.190000
41,Naples,40.835833,14.248611
42,Rome,41.893333,12.482778
